# Xarray-Spatial Bilateral: Edge-preserving smoothing

Raster data from remote sensing and field surveys picks up noise from sensors, atmospheric interference, and interpolation artifacts. Standard mean filters remove that noise but blur edges in the process. The bilateral filter handles both problems: it smooths flat regions while keeping sharp boundaries intact. That makes it useful as a preprocessing step before slope, aspect, or classification workflows where blurred edges would distort the results.

### What you'll build

1. Generate a noisy synthetic terrain
2. Compare bilateral filtering against the standard mean filter
3. Explore how `sigma_range` controls edge sensitivity
4. Test edge preservation on a synthetic step function

![Bilateral filter preview](images/bilateral_filter_preview.png)

[Terrain data](#Terrain-data) · [Bilateral vs. mean filter](#Bilateral-vs.-mean-filter) · [Effect of sigma_range](#Effect-of-sigma_range) · [Step-edge preservation](#Step-edge-preservation)

Standard imports plus `bilateral` and `mean` from xrspatial.

In [ ]:
import numpy as np
import xarray as xr

import matplotlib
import matplotlib.pyplot as plt
from matplotlib.patches import Patch

import xrspatial
from xrspatial import bilateral, mean, hillshade
from xrspatial.terrain import generate_terrain

## Terrain data

Synthetic elevation from `generate_terrain`, with Gaussian noise added on top. The same noisy raster is reused in every section below.

In [ ]:
W, H = 600, 400
canvas = xr.DataArray(np.zeros((H, W)), dims=['y', 'x'])
terrain = canvas.xrs.generate_terrain(seed=42)
illuminated = hillshade(terrain)

# Add Gaussian noise to simulate sensor artifacts
rng = np.random.default_rng(123)
noisy_terrain = terrain.copy(data=terrain.values + rng.normal(0, 15, terrain.shape))

terrain.plot.imshow(cmap='terrain', size=7.5, aspect=W/H, add_colorbar=False)

Clean terrain on the left, noisy version on the right. The noise adds about 15 units of standard deviation to each cell, enough to visibly roughen the surface and obscure fine detail.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
for ax, data, label in zip(axes,
                           [terrain, noisy_terrain],
                           ['Clean terrain', 'Noisy terrain']):
    illuminated.plot.imshow(ax=ax, cmap='gray', add_colorbar=False)
    data.plot.imshow(ax=ax, cmap='terrain', alpha=160/255, add_colorbar=False)
    ax.set_title(label, fontsize=13)
    ax.set_axis_off()
plt.tight_layout()

## Bilateral vs. mean filter

The [bilateral filter](https://en.wikipedia.org/wiki/Bilateral_filter) weights each neighbor by two Gaussians: one for spatial distance (`sigma_spatial`) and one for value similarity (`sigma_range`). Neighbors that are spatially close but have very different values get downweighted, so edges survive the smoothing. A standard mean filter weights all neighbors equally, which blurs everything.

Three panels below show the noisy input, the result of three mean-filter passes, and a single bilateral pass. Look at the ridge boundaries to see the difference.

In [ ]:
smoothed_bilateral = bilateral(noisy_terrain, sigma_spatial=2.0, sigma_range=20.0)
smoothed_mean = mean(noisy_terrain, passes=3)

fig, axes = plt.subplots(1, 3, figsize=(18, 5))
for ax, data, label in zip(axes,
                           [noisy_terrain, smoothed_mean, smoothed_bilateral],
                           ['Noisy input', 'Mean filter (3 passes)', 'Bilateral filter']):
    illuminated.plot.imshow(ax=ax, cmap='gray', add_colorbar=False)
    data.plot.imshow(ax=ax, cmap='terrain', alpha=160/255, add_colorbar=False)
    ax.set_title(label, fontsize=13)
    ax.set_axis_off()
plt.tight_layout()

The mean filter smooths the noise but softens ridge lines and valley edges. The bilateral filter removes a similar amount of noise while keeping those transitions sharp.

<div class="alert alert-block alert-warning">
<b>Edge effects.</b> The bilateral filter uses a kernel of radius <code>ceil(2 * sigma_spatial)</code>. Pixels near the raster boundary have fewer neighbors, which can introduce artifacts along the edges. Set <code>boundary='nearest'</code> or <code>boundary='reflect'</code> to pad the border instead of the default NaN fill.
</div>

## Effect of sigma_range

`sigma_range` controls how much value difference the filter tolerates before downweighting a neighbor. A small value (5.0) preserves nearly all edges but removes less noise. A large value (100.0) treats most neighbors as similar, making the bilateral filter behave like a standard Gaussian blur. The middle ground (20.0) works well for typical terrain noise.

Three panels below hold `sigma_spatial` fixed at 2.0 and vary `sigma_range`.

In [ ]:
sigma_ranges = [5.0, 20.0, 100.0]

fig, axes = plt.subplots(1, len(sigma_ranges), figsize=(18, 5))
for ax, sr in zip(axes, sigma_ranges):
    result = bilateral(noisy_terrain, sigma_spatial=2.0, sigma_range=sr)
    illuminated.plot.imshow(ax=ax, cmap='gray', add_colorbar=False)
    result.plot.imshow(ax=ax, cmap='terrain', alpha=160/255, add_colorbar=False)
    ax.set_title(f'sigma_range = {sr}', fontsize=13)
    ax.set_axis_off()
plt.tight_layout()

At `sigma_range=5.0`, ridgelines stay crisp but some speckle noise remains. At `sigma_range=100.0`, the result looks like a Gaussian blur with softened edges everywhere.

<div class="alert alert-block alert-info">
<b>Choosing sigma_range.</b> A good starting point is the standard deviation of the noise you want to remove. If you know your sensor adds ~15 units of noise, set <code>sigma_range</code> to roughly 15-25. Values much larger than the actual noise level will blur edges unnecessarily.
</div>

## Step-edge preservation

Terrain is complex, so it helps to test on something simple. This section creates a synthetic raster with a sharp vertical edge (0 on the left, 100 on the right) plus Gaussian noise. The bilateral filter should recover a clean step; the mean filter will produce a gradual ramp across the boundary.

The first row shows the three rasters. The second plot shows a cross-section through row 25, where you can see the transition profile directly.

In [ ]:
step = np.zeros((50, 100))
step[:, 50:] = 100.0
step_noisy = step + rng.normal(0, 5, step.shape)
step_agg = xr.DataArray(step_noisy, dims=['y', 'x'])

step_bilateral = bilateral(step_agg, sigma_spatial=2.0, sigma_range=10.0)
step_mean = mean(step_agg, passes=3)

fig, axes = plt.subplots(1, 3, figsize=(15, 4))
for ax, data, label in zip(axes,
                           [step_agg, step_mean, step_bilateral],
                           ['Noisy step edge', 'Mean filter', 'Bilateral filter']):
    data.plot.imshow(ax=ax, cmap='gray', add_colorbar=False)
    ax.set_title(label, fontsize=13)
    ax.set_axis_off()
plt.tight_layout()

In [ ]:
row = 25
fig, ax = plt.subplots(figsize=(10, 4))
ax.plot(step_agg.values[row], label='Noisy', alpha=0.5, color='gray')
ax.plot(step_mean.values[row], label='Mean filter', linewidth=2, color='steelblue')
ax.plot(step_bilateral.values[row], label='Bilateral filter', linewidth=2, color='darkorange')
ax.legend(fontsize=11, framealpha=0.9)
ax.set_xlabel('Column')
ax.set_ylabel('Value')
ax.set_title('Cross-section at row 25', fontsize=13)
plt.tight_layout()

The orange bilateral line holds a near-vertical transition at column 50, while the blue mean-filter line spreads the edge across roughly 10 columns. Both flatten out the noise on the flat portions, but only the bilateral filter keeps the boundary intact.

In [ ]:
# Save preview image for the "What you'll build" cell
import os as _os
_img_dir = _os.path.join(_os.getcwd(), 'images')
_os.makedirs(_img_dir, exist_ok=True)

_prev_backend = matplotlib.get_backend()
matplotlib.use('Agg')

fig, axes = plt.subplots(1, 3, figsize=(18, 5))
for ax, data, label in zip(axes,
                           [noisy_terrain, smoothed_mean, smoothed_bilateral],
                           ['Noisy input', 'Mean filter (3 passes)', 'Bilateral filter']):
    illuminated.plot.imshow(ax=ax, cmap='gray', add_colorbar=False)
    data.plot.imshow(ax=ax, cmap='terrain', alpha=160/255, add_colorbar=False)
    ax.set_title(label, fontsize=13)
    ax.set_axis_off()
plt.tight_layout()
fig.savefig(_os.path.join(_img_dir, 'bilateral_filter_preview.png'),
            bbox_inches='tight', dpi=120)
plt.close(fig)

try:
    matplotlib.use(_prev_backend)
except Exception:
    pass

### References

- [Bilateral filter](https://en.wikipedia.org/wiki/Bilateral_filter), Wikipedia
- Tomasi & Manduchi, [Bilateral Filtering for Gray and Color Images](https://users.soe.ucsc.edu/~manduchi/Papers/ICCV98.pdf), ICCV 1998
- [xrspatial bilateral API docs](https://makepath.github.io/xarray-spatial/reference/_autosummary/xrspatial.bilateral.html)